# C2 — Laptop Pilot Dataset Preparation and EDA

**Component 2:** Helpfulness-Guided Review Evidence Selection with Minority Preservation

This notebook implements the clean final PP1 laptop-pilot workflow.

## Final workflow

1. Discover the official Electronics metadata and review parquet shards.
2. Scan the full available Electronics metadata collection.
3. Identify laptop/notebook products using structured metadata.
4. Exclude obvious accessories and replacement parts.
5. Shortlist candidates using metadata activity (`rating_number`) only.
6. Process every Electronics review parquet shard to verify review availability.
7. Freeze 10 final laptop products without using helpful-vote outcomes.
8. Process every review shard again for only those final 10 products.
9. Collect all available reviews for the final products.
10. Select a deterministic chronological time-spread sample of 100 reviews per product.
11. Apply the same cleaning, stable-ID, usefulness-target, and chronological-split
    protocol used by the smartphone pilot.
12. Save reproducible outputs and run final validation.

## Research boundary

This remains a controlled PP1 development pilot.
It is not the final category-scale research evaluation dataset.

Helpful-vote outcomes are not used for laptop-product selection.


## 1. Environment, Imports, and Paths


In [2]:
from collections import defaultdict
from pathlib import Path

import hashlib
import json
import math
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from huggingface_hub import (
    HfApi,
    hf_hub_url,
)

from scipy.stats import (
    skew,
    spearmanr,
)


pd.set_option(
    "display.max_columns",
    60,
)

pd.set_option(
    "display.max_colwidth",
    140,
)


def find_project_root(
    start: Path
) -> Path:

    for candidate in [
        start,
        *start.parents,
    ]:

        if (
            (
                candidate
                / "data"
            ).exists()
            and
            (
                candidate
                / "notebooks"
            ).exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate Component 2 root."
    )


PROJECT_ROOT = (
    find_project_root(
        Path.cwd().resolve()
    )
)


PILOT_DIR = (
    PROJECT_ROOT
    / "data"
    / "pilot"
)


RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "eda"
    / "laptop"
)


FIGURES_DIR = (
    RESULTS_DIR
    / "figures"
)


CHECKPOINT_DIR = (
    RESULTS_DIR
    / "checkpoints_final_v4"
)


for directory in [
    PILOT_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    CHECKPOINT_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


RAW_PILOT_PATH = (
    PILOT_DIR
    / "laptop_pilot_raw.parquet"
)


CLEANED_PILOT_PATH = (
    PILOT_DIR
    / "laptop_pilot_cleaned.parquet"
)


print(
    "Project root:",
    PROJECT_ROOT,
)

print(
    "Results directory:",
    RESULTS_DIR,
)

print(
    "Checkpoint directory:",
    CHECKPOINT_DIR,
)


Project root: D:\Research\Repository\J26-DS-349\component_2
Results directory: D:\Research\Repository\J26-DS-349\component_2\results\eda\laptop
Checkpoint directory: D:\Research\Repository\J26-DS-349\component_2\results\eda\laptop\checkpoints_final_v4


## 2. Dataset and Pilot Configuration


In [3]:
PIPELINE_VERSION = (
    "laptop_pp1_final_v4"
)


DATASET_ID = (
    "McAuley-Lab/"
    "Amazon-Reviews-2023"
)


META_DIRECTORY = (
    "raw_meta_Electronics"
)


REVIEW_DIRECTORY = (
    "raw_review_Electronics"
)


REVIEW_FALLBACK_REVISION = (
    "32b99ef490c11e6d63223d962253bc696bdae583"
)


TARGET_PILOT_PRODUCTS = 10

REVIEW_CANDIDATE_POOL = 200

# Discovery pass only.
DISCOVERY_REVIEW_CAP = 100

MIN_REVIEWS_FOR_PILOT = 60

# Final raw pilot.
FINAL_RAW_REVIEWS_PER_PRODUCT = 100


MAX_RETRIES_PER_SHARD = 3

RETRY_SLEEP_SECONDS = 3


RAW_REVIEW_COLUMNS = [
    "rating",
    "title",
    "text",
    "asin",
    "parent_asin",
    "timestamp",
    "helpful_vote",
    "verified_purchase",
]


META_COLUMNS = [
    "parent_asin",
    "title",
    "average_rating",
    "rating_number",
    "categories",
    "main_category",
]


print(
    "Pipeline:",
    PIPELINE_VERSION,
)

print(
    "Target products:",
    TARGET_PILOT_PRODUCTS,
)

print(
    "Discovery candidates:",
    REVIEW_CANDIDATE_POOL,
)

print(
    "Final raw reviews/product:",
    FINAL_RAW_REVIEWS_PER_PRODUCT,
)


Pipeline: laptop_pp1_final_v4
Target products: 10
Discovery candidates: 200
Final raw reviews/product: 100


## 3. Discover Official Electronics Parquet Shards


In [4]:
api = HfApi()


main_files = (
    api.list_repo_files(
        repo_id=DATASET_ID,
        repo_type="dataset",
        revision="main",
    )
)


meta_repo_files = sorted(
    [
        file_name

        for file_name
        in main_files

        if (
            file_name.startswith(
                f"{META_DIRECTORY}/"
            )
            and
            file_name.endswith(
                ".parquet"
            )
        )
    ]
)


META_REVISION = "main"


review_repo_files = sorted(
    [
        file_name

        for file_name
        in main_files

        if (
            file_name.startswith(
                f"{REVIEW_DIRECTORY}/"
            )
            and
            file_name.endswith(
                ".parquet"
            )
        )
    ]
)


REVIEW_REVISION = "main"


if not review_repo_files:

    fallback_files = (
        api.list_repo_files(
            repo_id=DATASET_ID,
            repo_type="dataset",
            revision=(
                REVIEW_FALLBACK_REVISION
            ),
        )
    )


    review_repo_files = sorted(
        [
            file_name

            for file_name
            in fallback_files

            if (
                file_name.startswith(
                    f"{REVIEW_DIRECTORY}/"
                )
                and
                file_name.endswith(
                    ".parquet"
                )
            )
        ]
    )


    REVIEW_REVISION = (
        REVIEW_FALLBACK_REVISION
    )


if not meta_repo_files:

    raise RuntimeError(
        "No Electronics metadata "
        "parquet shards found."
    )


if not review_repo_files:

    raise RuntimeError(
        "No Electronics review parquet "
        "shards found."
    )


print(
    "Metadata shards:",
    len(
        meta_repo_files
    ),
)


print(
    "Metadata revision:",
    META_REVISION,
)


print(
    "Review shards:",
    len(
        review_repo_files
    ),
)


print(
    "Review revision:",
    REVIEW_REVISION,
)


def dataset_file_url(
    repo_file,
    revision,
):

    return hf_hub_url(
        repo_id=DATASET_ID,
        filename=repo_file,
        repo_type="dataset",
        revision=revision,
    )


Metadata shards: 10
Metadata revision: main
Review shards: 34
Review revision: 32b99ef490c11e6d63223d962253bc696bdae583


## 4. Structured Laptop Product Identification


In [5]:
LAPTOP_CATEGORY_TERMS = [
    "laptop",
    "notebook",
    "chromebook",
    "macbook",
]


CATEGORY_ACCESSORY_TERMS = [
    "accessories",
    "accessory",
    "case",
    "cases",
    "sleeve",
    "sleeves",
    "bag",
    "bags",
    "backpack",
    "charger",
    "chargers",
    "adapter",
    "adapters",
    "stand",
    "stands",
    "dock",
    "docks",
    "docking",
    "battery",
    "batteries",
    "replacement",
    "screen protector",
    "keyboard cover",
    "cooling pad",
    "cooler",
    "parts",
    "components",
]


TITLE_ACCESSORY_PATTERNS = [
    r"\bcharger\b",
    r"\bpower adapter\b",
    r"\bac adapter\b",
    r"\breplacement adapter\b",
    r"\bcase\b",
    r"\bcover\b",
    r"\bsleeve\b",
    r"\blaptop bag\b",
    r"\bbackpack\b",
    r"\bstand\b",
    r"\bdock\b",
    r"\bdocking station\b",
    r"\bcooling pad\b",
    r"\bcooler\b",
    r"\bscreen protector\b",
    r"\breplacement screen\b",
    r"\blcd replacement\b",
    r"\bkeyboard cover\b",
    r"\bkeyboard skin\b",
    r"\breplacement battery\b",
    r"\blaptop battery\b",
    r"\bprivacy screen\b",
    r"\bprotective shell\b",
    r"\bhinge replacement\b",
]


def normalize_categories(
    value
):

    if isinstance(
        value,
        np.ndarray,
    ):
        value = value.tolist()


    if not isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        return []


    return [
        str(item).strip()

        for item
        in value

        if str(item).strip()
    ]


def has_laptop_signal(
    categories
):

    labels = (
        normalize_categories(
            categories
        )
    )


    return any(

        any(
            term
            in
            label.lower()

            for term
            in
            LAPTOP_CATEGORY_TERMS
        )

        for label
        in labels
    )


def accessory_only_laptop_category(
    categories
):

    labels = (
        normalize_categories(
            categories
        )
    )


    laptop_labels = [

        label.lower()

        for label
        in labels

        if any(
            term
            in
            label.lower()

            for term
            in
            LAPTOP_CATEGORY_TERMS
        )
    ]


    if not laptop_labels:

        return False


    return all(

        any(
            term
            in
            label

            for term
            in
            CATEGORY_ACCESSORY_TERMS
        )

        for label
        in laptop_labels
    )


def obvious_accessory_title(
    title
):

    text = (
        ""
        if title is None
        else str(
            title
        ).lower()
    )


    return any(

        re.search(
            pattern,
            text,
            flags=re.IGNORECASE,
        )

        for pattern
        in TITLE_ACCESSORY_PATTERNS
    )


def is_laptop_metadata_row(
    row
):

    if row.get(
        "parent_asin"
    ) is None:

        return False


    if not has_laptop_signal(
        row.get(
            "categories"
        )
    ):

        return False


    if accessory_only_laptop_category(
        row.get(
            "categories"
        )
    ):

        return False


    if obvious_accessory_title(
        row.get(
            "title"
        )
    ):

        return False


    return True


## 5. Full Electronics Metadata Scan

Every metadata parquet shard is processed.

Per-shard cache files allow a rerun to resume without repeating
completed metadata work.


In [6]:
META_CACHE_DIR = (
    CHECKPOINT_DIR
    / "metadata_candidates"
)


META_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


metadata_parts = []

metadata_diagnostics = []


for (
    shard_index,
    repo_file,
) in enumerate(
    meta_repo_files,
    start=1,
):

    cache_path = (
        META_CACHE_DIR
        /
        (
            f"meta_candidates_"
            f"{shard_index:02d}.parquet"
        )
    )


    if cache_path.exists():

        part = (
            pd.read_parquet(
                cache_path
            )
        )


        metadata_parts.append(
            part
        )


        metadata_diagnostics.append(
            {
                "shard":
                    repo_file,

                "status":
                    "loaded_from_cache",

                "candidate_rows":
                    len(
                        part
                    ),
            }
        )


        print(
            f"Metadata shard "
            f"{shard_index}/"
            f"{len(meta_repo_files)} "
            f"loaded from cache."
        )


        continue


    print()

    print(
        f"Scanning metadata shard "
        f"{shard_index}/"
        f"{len(meta_repo_files)}"
    )


    success = False

    last_error = None


    for attempt in range(
        1,
        MAX_RETRIES_PER_SHARD
        +
        1,
    ):

        try:

            stream = load_dataset(
                "parquet",
                data_files=(
                    dataset_file_url(
                        repo_file,
                        META_REVISION,
                    )
                ),
                split="train",
                streaming=True,
            )


            try:

                stream = (
                    stream.select_columns(
                        META_COLUMNS
                    )
                )

            except Exception:

                pass


            rows_scanned = 0

            rows = []


            for row in stream:

                rows_scanned += 1


                if (
                    is_laptop_metadata_row(
                        row
                    )
                ):

                    rows.append(
                        {
                            column:
                                row.get(
                                    column
                                )

                            for column
                            in META_COLUMNS
                        }
                    )


            part = pd.DataFrame(
                rows,
                columns=META_COLUMNS,
            )


            part.to_parquet(
                cache_path,
                index=False,
            )


            metadata_parts.append(
                part
            )


            metadata_diagnostics.append(
                {
                    "shard":
                        repo_file,

                    "status":
                        "processed",

                    "rows_scanned":
                        rows_scanned,

                    "candidate_rows":
                        len(
                            part
                        ),
                }
            )


            print(
                "Rows scanned:",
                f"{rows_scanned:,}",
            )


            print(
                "Laptop candidates:",
                len(
                    part
                ),
            )


            success = True

            break


        except Exception as error:

            last_error = error


            print(
                f"Attempt "
                f"{attempt}/"
                f"{MAX_RETRIES_PER_SHARD} failed:"
            )


            print(
                f"{type(error).__name__}: "
                f"{error}"
            )


            if (
                attempt
                <
                MAX_RETRIES_PER_SHARD
            ):

                time.sleep(
                    RETRY_SLEEP_SECONDS
                )


    if not success:

        raise RuntimeError(
            "Metadata shard failed: "
            f"{repo_file}"
        ) from last_error


laptop_candidates = (
    pd.concat(
        metadata_parts,
        ignore_index=True,
    )
)


laptop_candidates[
    "parent_asin"
] = (
    laptop_candidates[
        "parent_asin"
    ]
    .astype(str)
)


laptop_candidates[
    "rating_number"
] = (
    pd.to_numeric(
        laptop_candidates[
            "rating_number"
        ],
        errors="coerce",
    )
    .fillna(0)
)


laptop_candidates[
    "average_rating"
] = pd.to_numeric(
    laptop_candidates[
        "average_rating"
    ],
    errors="coerce",
)


laptop_candidates = (

    laptop_candidates

    .sort_values(
        [
            "rating_number",
            "parent_asin",
        ],
        ascending=[
            False,
            True,
        ],
    )

    .drop_duplicates(
        "parent_asin",
        keep="first",
    )

    .reset_index(
        drop=True
    )
)


FULL_METADATA_CANDIDATE_PATH = (
    RESULTS_DIR
    /
    "laptop_full_metadata_candidates.csv"
)


laptop_candidates.to_csv(
    FULL_METADATA_CANDIDATE_PATH,
    index=False,
)


METADATA_DIAGNOSTICS_PATH = (
    RESULTS_DIR
    /
    "laptop_metadata_scan_diagnostics.csv"
)


pd.DataFrame(
    metadata_diagnostics
).to_csv(
    METADATA_DIAGNOSTICS_PATH,
    index=False,
)


print()

print(
    "Unique metadata-qualified "
    "laptop products:",
    laptop_candidates[
        "parent_asin"
    ].nunique(),
)


display(
    laptop_candidates[
        [
            "parent_asin",
            "title",
            "average_rating",
            "rating_number",
            "categories",
        ]
    ]
    .head(
        30
    )
)


Metadata shard 1/10 loaded from cache.
Metadata shard 2/10 loaded from cache.
Metadata shard 3/10 loaded from cache.
Metadata shard 4/10 loaded from cache.
Metadata shard 5/10 loaded from cache.
Metadata shard 6/10 loaded from cache.
Metadata shard 7/10 loaded from cache.
Metadata shard 8/10 loaded from cache.
Metadata shard 9/10 loaded from cache.
Metadata shard 10/10 loaded from cache.

Unique metadata-qualified laptop products: 32877


,parent_asin,title,average_rating,rating_number,categories
0,B0C2ZMJW53,"Acer Aspire 3 A314-23P-R3QA Slim Laptop | 14.0"" Full HD IPS Display | AMD Ryzen 5 7520U Quad-Core Processor | AMD Radeon Graphics | 8GB ...",4.5,35925,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"
1,B07GM2J11Q,"Lenovo Chromebook C330 2-in-1 Convertible Laptop, 11.6"" HD Display, MediaTek MT8173C, 4GB RAM, 64GB Storage, Chrome OS, Blizzard White",4.5,10961,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"
2,B08GG9QKX6,"Apple MacBook Air (13-inch Retina Display, 8GB RAM, 256GB SSD Storage) - Gold (Previous Model)",4.8,8799,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"
3,B081V6W99V,"[2021 Version]ASUS Vivobook Laptop L210 11.6” ultra thin, Intel Celeron N4020 Processor, 4GB RAM, 64GB eMMC storage, Windows 10 Home in ...",4.3,8402,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"
4,B0C7H1XD7V,"Lenovo Flex 5 14 2-in-1 Laptop, 14.0"" FHD Touch Display, AMD Ryzen 5 4500U, 16GB RAM, 256GB Storage, AMD Radeon Graphics, Digital Pen In...",4.4,6964,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, 2 in 1 Laptops]"
5,B07N6S4SY1,"ASUS VivoBook L203MA Ultra-Thin Laptop, Intel Celeron N4000 Processor, 4GB LPDDR4, 64GB eMMC, 11.6” HD, USB-C, Windows 10 in S Mode (Swi...",4.2,6468,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"
6,B096JQY4YR,"HP Chromebook 14-inch HD Touchscreen Laptop, Intel Celeron N4000, 4 GB RAM, 32 GB eMMC, Chrome (14a-na0030nr, Mineral Silver)",4.5,6280,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"
7,B009T3EYHO,"Samsung Chromebook (Wi-Fi, 11.6-Inch) 2012 Model",4.1,6054,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"
8,B0869L1326,"ASUS VivoBook 15 Thin and Light Laptop, 15.6” FHD Display, Intel i3-1005G1 CPU, 8GB RAM, 128GB SSD, Backlit Keyboard, Fingerprint, Windo...",4.5,5851,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"
9,B01K1IO3QW,"Acer Aspire E 15 E5-575-33BM 15.6-Inch Full HD Notebook (Intel Core i3-7100U Processor 7th Generation , 4GB DDR4, 1TB 5400RPM Hard Drive...",3.9,5711,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"


## 6. Metadata-Based Discovery Candidate Pool


In [7]:
review_candidate_meta = (

    laptop_candidates

    .sort_values(
        [
            "rating_number",
            "parent_asin",
        ],
        ascending=[
            False,
            True,
        ],
    )

    .head(
        REVIEW_CANDIDATE_POOL
    )

    .copy()
)


review_candidate_ids = set(
    review_candidate_meta[
        "parent_asin"
    ]
    .astype(str)
)


candidate_signature = (

    hashlib.sha256(

        (
            PIPELINE_VERSION
            +
            "|DISCOVERY|"
            +
            "|".join(
                sorted(
                    review_candidate_ids
                )
            )
        )
        .encode(
            "utf-8"
        )

    )
    .hexdigest()[:16]
)


print(
    "Discovery candidate products:",
    len(
        review_candidate_ids
    ),
)


print(
    "Discovery signature:",
    candidate_signature,
)


Discovery candidate products: 200
Discovery signature: 8d4ce10c0efd6623


## 7. Complete Review-Availability Discovery Pass

Every Electronics review parquet shard is processed.

Up to 100 reviews per candidate are retained in this discovery pass.
This cap is only used to establish whether sufficient review evidence exists.


In [8]:
DISCOVERY_PATH = (
    CHECKPOINT_DIR
    /
    "discovery_reviews.parquet"
)


DISCOVERY_STATE_PATH = (
    CHECKPOINT_DIR
    /
    "discovery_state.json"
)


DISCOVERY_DIAGNOSTICS_PATH = (
    RESULTS_DIR
    /
    "laptop_discovery_review_shard_diagnostics.csv"
)


def empty_review_frame():

    return pd.DataFrame(
        columns=RAW_REVIEW_COLUMNS
    )


if (
    DISCOVERY_PATH.exists()
    and
    DISCOVERY_STATE_PATH.exists()
):

    state = json.loads(
        DISCOVERY_STATE_PATH.read_text(
            encoding="utf-8"
        )
    )


    if (
        state.get(
            "candidate_signature"
        )
        ==
        candidate_signature
    ):

        discovery_checkpoint = (
            pd.read_parquet(
                DISCOVERY_PATH
            )
        )


        completed_discovery_shards = set(
            state.get(
                "completed_shards",
                [],
            )
        )


        discovery_diagnostics = (
            state.get(
                "diagnostics",
                [],
            )
        )


    else:

        discovery_checkpoint = (
            empty_review_frame()
        )

        completed_discovery_shards = set()

        discovery_diagnostics = []


else:

    discovery_checkpoint = (
        empty_review_frame()
    )

    completed_discovery_shards = set()

    discovery_diagnostics = []


discovery_buffers = defaultdict(
    list
)


if not discovery_checkpoint.empty:

    for row in (
        discovery_checkpoint
        .to_dict(
            orient="records"
        )
    ):

        product = str(
            row[
                "parent_asin"
            ]
        )


        if (
            product
            in
            review_candidate_ids
            and
            len(
                discovery_buffers[
                    product
                ]
            )
            <
            DISCOVERY_REVIEW_CAP
        ):

            discovery_buffers[
                product
            ].append(
                row
            )


def flatten_discovery():

    rows = []


    for product in sorted(
        discovery_buffers
    ):

        rows.extend(
            discovery_buffers[
                product
            ]
        )


    if not rows:

        return empty_review_frame()


    return pd.DataFrame(
        rows,
        columns=RAW_REVIEW_COLUMNS,
    )


def save_discovery():

    frame = (
        flatten_discovery()
    )


    frame.to_parquet(
        DISCOVERY_PATH,
        index=False,
    )


    state = {
        "pipeline_version":
            PIPELINE_VERSION,

        "candidate_signature":
            candidate_signature,

        "completed_shards":
            sorted(
                completed_discovery_shards
            ),

        "diagnostics":
            discovery_diagnostics,
    }


    DISCOVERY_STATE_PATH.write_text(
        json.dumps(
            state,
            indent=2,
        ),
        encoding="utf-8",
    )


for (
    shard_index,
    repo_file,
) in enumerate(
    review_repo_files,
    start=1,
):

    if (
        repo_file
        in
        completed_discovery_shards
    ):

        print(
            f"Skipping discovery shard "
            f"{shard_index}/"
            f"{len(review_repo_files)}"
        )


        continue


    print()

    print(
        f"Discovery shard "
        f"{shard_index}/"
        f"{len(review_repo_files)}"
    )


    success = False

    last_error = None


    for attempt in range(
        1,
        MAX_RETRIES_PER_SHARD
        +
        1,
    ):

        try:

            stream = load_dataset(
                "parquet",
                data_files=(
                    dataset_file_url(
                        repo_file,
                        REVIEW_REVISION,
                    )
                ),
                split="train",
                streaming=True,
            )


            try:

                stream = (
                    stream.select_columns(
                        RAW_REVIEW_COLUMNS
                    )
                )

            except Exception:

                pass


            rows_scanned = 0

            rows_matched = 0

            local_rows = defaultdict(
                list
            )


            working_counts = {

                product:
                    len(
                        discovery_buffers[
                            product
                        ]
                    )

                for product
                in review_candidate_ids
            }


            for review in stream:

                rows_scanned += 1


                product = (
                    review.get(
                        "parent_asin"
                    )
                )


                if product is None:

                    continue


                product = str(
                    product
                )


                if (
                    product
                    not in
                    review_candidate_ids
                ):

                    continue


                rows_matched += 1


                if (
                    working_counts[
                        product
                    ]
                    >=
                    DISCOVERY_REVIEW_CAP
                ):

                    continue


                local_rows[
                    product
                ].append(
                    {

                        column:
                            review.get(
                                column
                            )

                        for column
                        in RAW_REVIEW_COLUMNS
                    }
                )


                working_counts[
                    product
                ] += 1


            rows_added = 0


            for (
                product,
                rows,
            ) in local_rows.items():

                discovery_buffers[
                    product
                ].extend(
                    rows
                )


                rows_added += len(
                    rows
                )


            completed_discovery_shards.add(
                repo_file
            )


            counts = {

                product:
                    len(
                        discovery_buffers[
                            product
                        ]
                    )

                for product
                in review_candidate_ids
            }


            qualified = sum(

                count
                >=
                MIN_REVIEWS_FOR_PILOT

                for count
                in counts.values()
            )


            capped = sum(

                count
                >=
                DISCOVERY_REVIEW_CAP

                for count
                in counts.values()
            )


            discovery_diagnostics.append(
                {
                    "review_shard":
                        repo_file,

                    "rows_scanned":
                        rows_scanned,

                    "candidate_rows_matched":
                        rows_matched,

                    "candidate_rows_added":
                        rows_added,

                    "qualified_products":
                        qualified,

                    "products_at_discovery_cap":
                        capped,
                }
            )


            save_discovery()


            print(
                "Rows scanned:",
                f"{rows_scanned:,}",
            )


            print(
                "Matched:",
                f"{rows_matched:,}",
            )


            print(
                "Added:",
                f"{rows_added:,}",
            )


            print(
                f"Products >= "
                f"{MIN_REVIEWS_FOR_PILOT}:",
                qualified,
            )


            success = True

            break


        except Exception as error:

            last_error = error


            print(
                f"Attempt "
                f"{attempt}/"
                f"{MAX_RETRIES_PER_SHARD} failed:"
            )


            print(
                error
            )


            if (
                attempt
                <
                MAX_RETRIES_PER_SHARD
            ):

                time.sleep(
                    RETRY_SLEEP_SECONDS
                )


    if not success:

        raise RuntimeError(
            "Discovery shard failed: "
            f"{repo_file}"
        ) from last_error


assert (
    len(
        completed_discovery_shards
    )
    ==
    len(
        review_repo_files
    )
)


pd.DataFrame(
    discovery_diagnostics
).to_csv(
    DISCOVERY_DIAGNOSTICS_PATH,
    index=False,
)


print()

print(
    "Discovery review pass complete."
)


Skipping discovery shard 1/34
Skipping discovery shard 2/34
Skipping discovery shard 3/34
Skipping discovery shard 4/34
Skipping discovery shard 5/34
Skipping discovery shard 6/34
Skipping discovery shard 7/34
Skipping discovery shard 8/34
Skipping discovery shard 9/34
Skipping discovery shard 10/34
Skipping discovery shard 11/34
Skipping discovery shard 12/34
Skipping discovery shard 13/34
Skipping discovery shard 14/34
Skipping discovery shard 15/34
Skipping discovery shard 16/34
Skipping discovery shard 17/34
Skipping discovery shard 18/34
Skipping discovery shard 19/34
Skipping discovery shard 20/34
Skipping discovery shard 21/34
Skipping discovery shard 22/34
Skipping discovery shard 23/34
Skipping discovery shard 24/34
Skipping discovery shard 25/34
Skipping discovery shard 26/34
Skipping discovery shard 27/34
Skipping discovery shard 28/34
Skipping discovery shard 29/34
Skipping discovery shard 30/34
Skipping discovery shard 31/34
Skipping discovery shard 32/34
Skipping discover

## 8. Freeze Final 10 Laptop Products


In [9]:
discovery_counts = pd.DataFrame(
    [
        {
            "parent_asin":
                product,

            "review_count_collected":
                len(
                    discovery_buffers[
                        product
                    ]
                ),
        }

        for product
        in sorted(
            review_candidate_ids
        )
    ]
)


availability = (

    review_candidate_meta[
        [
            "parent_asin",
            "title",
            "average_rating",
            "rating_number",
            "categories",
            "main_category",
        ]
    ]

    .copy()
)


availability[
    "parent_asin"
] = (
    availability[
        "parent_asin"
    ]
    .astype(str)
)


availability = (
    availability.merge(
        discovery_counts,
        on="parent_asin",
        how="left",
    )
)


availability[
    "review_count_collected"
] = (
    availability[
        "review_count_collected"
    ]
    .fillna(0)
    .astype(int)
)


eligible_products = (

    availability[

        availability[
            "review_count_collected"
        ]
        >=
        MIN_REVIEWS_FOR_PILOT
    ]

    .copy()
)


print(
    "Eligible laptop products:",
    len(
        eligible_products
    ),
)


if (
    len(
        eligible_products
    )
    <
    TARGET_PILOT_PRODUCTS
):

    raise RuntimeError(
        "Fewer than 10 laptop products "
        "have sufficient review availability."
    )


selected_pilot_products = (

    eligible_products

    .sort_values(
        [
            "rating_number",
            "review_count_collected",
            "parent_asin",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )

    .head(
        TARGET_PILOT_PRODUCTS
    )

    .copy()

    .reset_index(
        drop=True
    )
)


FINAL_LAPTOP_ASINS = set(

    selected_pilot_products[
        "parent_asin"
    ]
    .astype(str)
)


final_product_signature = (

    hashlib.sha256(

        (
            PIPELINE_VERSION
            +
            "|FINAL|"
            +
            "|".join(
                sorted(
                    FINAL_LAPTOP_ASINS
                )
            )
        )
        .encode(
            "utf-8"
        )

    )
    .hexdigest()[:16]
)


FINAL_PRODUCT_FREEZE_PATH = (
    RESULTS_DIR
    /
    "laptop_final_product_freeze.csv"
)


freeze_output = (
    selected_pilot_products
    .copy()
)


freeze_output[
    "categories"
] = (
    freeze_output[
        "categories"
    ]
    .apply(
        lambda value:

            json.dumps(
                (
                    value.tolist()
                    if isinstance(
                        value,
                        np.ndarray,
                    )
                    else value
                ),
                ensure_ascii=False,
            )

            if isinstance(
                value,
                (
                    list,
                    tuple,
                    np.ndarray,
                ),
            )

            else str(
                value
            )
    )
)


freeze_output.to_csv(
    FINAL_PRODUCT_FREEZE_PATH,
    index=False,
)


display(
    selected_pilot_products[
        [
            "parent_asin",
            "review_count_collected",
            "title",
            "average_rating",
            "rating_number",
            "categories",
        ]
    ]
)


print()

print(
    "MANUAL CHECK REQUIRED: "
    "Confirm that these 10 products "
    "are actual laptops/notebooks."
)


Eligible laptop products: 185


,parent_asin,review_count_collected,title,average_rating,rating_number,categories
0,B0C2ZMJW53,100,"Acer Aspire 3 A314-23P-R3QA Slim Laptop | 14.0"" Full HD IPS Display | AMD Ryzen 5 7520U Quad-Core Processor | AMD Radeon Graphics | 8GB ...",4.5,35925,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"
1,B07GM2J11Q,100,"Lenovo Chromebook C330 2-in-1 Convertible Laptop, 11.6"" HD Display, MediaTek MT8173C, 4GB RAM, 64GB Storage, Chrome OS, Blizzard White",4.5,10961,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"
2,B08GG9QKX6,100,"Apple MacBook Air (13-inch Retina Display, 8GB RAM, 256GB SSD Storage) - Gold (Previous Model)",4.8,8799,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"
3,B081V6W99V,100,"[2021 Version]ASUS Vivobook Laptop L210 11.6” ultra thin, Intel Celeron N4020 Processor, 4GB RAM, 64GB eMMC storage, Windows 10 Home in ...",4.3,8402,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"
4,B0C7H1XD7V,100,"Lenovo Flex 5 14 2-in-1 Laptop, 14.0"" FHD Touch Display, AMD Ryzen 5 4500U, 16GB RAM, 256GB Storage, AMD Radeon Graphics, Digital Pen In...",4.4,6964,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, 2 in 1 Laptops]"
5,B07N6S4SY1,100,"ASUS VivoBook L203MA Ultra-Thin Laptop, Intel Celeron N4000 Processor, 4GB LPDDR4, 64GB eMMC, 11.6” HD, USB-C, Windows 10 in S Mode (Swi...",4.2,6468,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"
6,B096JQY4YR,100,"HP Chromebook 14-inch HD Touchscreen Laptop, Intel Celeron N4000, 4 GB RAM, 32 GB eMMC, Chrome (14a-na0030nr, Mineral Silver)",4.5,6280,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"
7,B009T3EYHO,100,"Samsung Chromebook (Wi-Fi, 11.6-Inch) 2012 Model",4.1,6054,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"
8,B0869L1326,100,"ASUS VivoBook 15 Thin and Light Laptop, 15.6” FHD Display, Intel i3-1005G1 CPU, 8GB RAM, 128GB SSD, Backlit Keyboard, Fingerprint, Windo...",4.5,5851,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"
9,B01K1IO3QW,100,"Acer Aspire E 15 E5-575-33BM 15.6-Inch Full HD Notebook (Intel Core i3-7100U Processor 7th Generation , 4GB DDR4, 1TB 5400RPM Hard Drive...",3.9,5711,"[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]"



MANUAL CHECK REQUIRED: Confirm that these 10 products are actual laptops/notebooks.


## 9. Collect All Reviews for the Frozen Final 10 Products


In [10]:
FINAL_ALL_REVIEWS_PATH = (
    CHECKPOINT_DIR
    /
    "final10_all_reviews.parquet"
)


FINAL_STATE_PATH = (
    CHECKPOINT_DIR
    /
    "final10_state.json"
)


FINAL_REVIEW_DIAGNOSTICS_PATH = (
    RESULTS_DIR
    /
    "laptop_final10_review_shard_diagnostics.csv"
)


if (
    FINAL_ALL_REVIEWS_PATH.exists()
    and
    FINAL_STATE_PATH.exists()
):

    state = json.loads(
        FINAL_STATE_PATH.read_text(
            encoding="utf-8"
        )
    )


    if (
        state.get(
            "final_product_signature"
        )
        ==
        final_product_signature
    ):

        final_all_reviews = (
            pd.read_parquet(
                FINAL_ALL_REVIEWS_PATH
            )
        )


        completed_final_shards = set(
            state.get(
                "completed_shards",
                [],
            )
        )


        final_diagnostics = (
            state.get(
                "diagnostics",
                [],
            )
        )


    else:

        final_all_reviews = (
            empty_review_frame()
        )

        completed_final_shards = set()

        final_diagnostics = []


else:

    final_all_reviews = (
        empty_review_frame()
    )

    completed_final_shards = set()

    final_diagnostics = []


for (
    shard_index,
    repo_file,
) in enumerate(
    review_repo_files,
    start=1,
):

    if (
        repo_file
        in
        completed_final_shards
    ):

        print(
            f"Skipping final shard "
            f"{shard_index}/"
            f"{len(review_repo_files)}"
        )


        continue


    print()

    print(
        f"Final review extraction "
        f"{shard_index}/"
        f"{len(review_repo_files)}"
    )


    success = False

    last_error = None


    for attempt in range(
        1,
        MAX_RETRIES_PER_SHARD
        +
        1,
    ):

        try:

            stream = load_dataset(
                "parquet",
                data_files=(
                    dataset_file_url(
                        repo_file,
                        REVIEW_REVISION,
                    )
                ),
                split="train",
                streaming=True,
            )


            try:

                stream = (
                    stream.select_columns(
                        RAW_REVIEW_COLUMNS
                    )
                )

            except Exception:

                pass


            rows_scanned = 0

            matches = []


            for review in stream:

                rows_scanned += 1


                product = (
                    review.get(
                        "parent_asin"
                    )
                )


                if product is None:

                    continue


                product = str(
                    product
                )


                if (
                    product
                    not in
                    FINAL_LAPTOP_ASINS
                ):

                    continue


                matches.append(
                    {

                        column:
                            review.get(
                                column
                            )

                        for column
                        in RAW_REVIEW_COLUMNS
                    }
                )


            if matches:

                final_all_reviews = (
                    pd.concat(
                        [
                            final_all_reviews,
                            pd.DataFrame(
                                matches,
                                columns=RAW_REVIEW_COLUMNS,
                            ),
                        ],
                        ignore_index=True,
                    )
                )


            completed_final_shards.add(
                repo_file
            )


            final_diagnostics.append(
                {
                    "review_shard":
                        repo_file,

                    "rows_scanned":
                        rows_scanned,

                    "final_product_reviews_matched":
                        len(
                            matches
                        ),

                    "cumulative_final_reviews":
                        len(
                            final_all_reviews
                        ),
                }
            )


            final_all_reviews.to_parquet(
                FINAL_ALL_REVIEWS_PATH,
                index=False,
            )


            FINAL_STATE_PATH.write_text(
                json.dumps(
                    {
                        "pipeline_version":
                            PIPELINE_VERSION,

                        "final_product_signature":
                            final_product_signature,

                        "completed_shards":
                            sorted(
                                completed_final_shards
                            ),

                        "diagnostics":
                            final_diagnostics,
                    },
                    indent=2,
                ),
                encoding="utf-8",
            )


            print(
                "Rows scanned:",
                f"{rows_scanned:,}",
            )


            print(
                "Matched final reviews:",
                f"{len(matches):,}",
            )


            print(
                "Cumulative:",
                f"{len(final_all_reviews):,}",
            )


            success = True

            break


        except Exception as error:

            last_error = error


            print(
                f"Attempt "
                f"{attempt}/"
                f"{MAX_RETRIES_PER_SHARD} failed:"
            )


            print(
                error
            )


            if (
                attempt
                <
                MAX_RETRIES_PER_SHARD
            ):

                time.sleep(
                    RETRY_SLEEP_SECONDS
                )


    if not success:

        raise RuntimeError(
            "Final review shard failed: "
            f"{repo_file}"
        ) from last_error


assert (
    len(
        completed_final_shards
    )
    ==
    len(
        review_repo_files
    )
)


final_diagnostics_df = (
    pd.DataFrame(
        final_diagnostics
    )
)


final_diagnostics_df.to_csv(
    FINAL_REVIEW_DIAGNOSTICS_PATH,
    index=False,
)


print()

print(
    "Final full-review pass complete."
)


print(
    "All final-product reviews:",
    len(
        final_all_reviews
    ),
)


display(

    final_all_reviews[
        "parent_asin"
    ]
    .astype(str)
    .value_counts()
    .rename(
        "all_available_reviews"
    )
    .to_frame()
)


Skipping final shard 1/34
Skipping final shard 2/34
Skipping final shard 3/34
Skipping final shard 4/34
Skipping final shard 5/34
Skipping final shard 6/34
Skipping final shard 7/34
Skipping final shard 8/34
Skipping final shard 9/34
Skipping final shard 10/34
Skipping final shard 11/34
Skipping final shard 12/34
Skipping final shard 13/34
Skipping final shard 14/34
Skipping final shard 15/34

Final review extraction 16/34
Rows scanned: 1,290,793
Matched final reviews: 513
Cumulative: 8,410

Final review extraction 17/34
Rows scanned: 1,290,792
Matched final reviews: 525
Cumulative: 8,935

Final review extraction 18/34
Rows scanned: 1,290,792
Matched final reviews: 572
Cumulative: 9,507

Final review extraction 19/34
Rows scanned: 1,290,792
Matched final reviews: 554
Cumulative: 10,061

Final review extraction 20/34
Rows scanned: 1,290,792
Matched final reviews: 514
Cumulative: 10,575

Final review extraction 21/34
Rows scanned: 1,290,792
Matched final reviews: 2,279
Cumulative: 12,854

,all_available_reviews
parent_asin,
B009T3EYHO,5117
B0C2ZMJW53,5053
B01K1IO3QW,4056
B07GM2J11Q,1854
B0C7H1XD7V,1187
B081V6W99V,827
B07N6S4SY1,642
B08GG9QKX6,609
B0869L1326,557


## 10. Deterministic Time-Spread Sampling


In [11]:
TEMPORAL_SAMPLING_PATH = (
    RESULTS_DIR
    /
    "laptop_temporal_sampling_diagnostics.csv"
)


pool = (
    final_all_reviews
    .copy()
)


pool[
    "parent_asin"
] = (
    pool[
        "parent_asin"
    ]
    .astype(str)
)


pool[
    "_sample_timestamp"
] = pd.to_numeric(
    pool[
        "timestamp"
    ],
    errors="coerce",
)


pool[
    "_sample_date"
] = pd.to_datetime(
    pool[
        "_sample_timestamp"
    ],
    unit="ms",
    errors="coerce",
    utc=True,
)


sampled_parts = []

sampling_rows = []


for product in sorted(
    FINAL_LAPTOP_ASINS
):

    product_all = (

        pool[

            pool[
                "parent_asin"
            ]
            ==
            product
        ]

        .copy()
    )


    valid = (

        product_all[

            product_all[
                "_sample_date"
            ]
            .notna()
        ]

        .copy()
    )


    if (
        len(
            valid
        )
        <
        FINAL_RAW_REVIEWS_PER_PRODUCT
    ):

        raise RuntimeError(
            f"{product}: insufficient "
            "valid timestamp reviews."
        )


    valid[
        "_sort_asin"
    ] = (
        valid[
            "asin"
        ]
        .fillna("")
        .astype(str)
    )


    valid[
        "_sort_title"
    ] = (
        valid[
            "title"
        ]
        .fillna("")
        .astype(str)
    )


    valid[
        "_sort_text"
    ] = (
        valid[
            "text"
        ]
        .fillna("")
        .astype(str)
    )


    valid = (

        valid

        .sort_values(
            [
                "_sample_date",
                "_sort_asin",
                "_sort_title",
                "_sort_text",
            ],
            kind="mergesort",
        )

        .reset_index(
            drop=True
        )
    )


    positions = np.linspace(
        0,
        len(
            valid
        )
        -
        1,
        FINAL_RAW_REVIEWS_PER_PRODUCT,
        dtype=int,
    )


    if (
        len(
            np.unique(
                positions
            )
        )
        !=
        FINAL_RAW_REVIEWS_PER_PRODUCT
    ):

        raise RuntimeError(
            f"{product}: duplicate "
            "sample positions."
        )


    sample = (
        valid.iloc[
            positions
        ]
        .copy()
    )


    sampled_parts.append(
        sample[
            RAW_REVIEW_COLUMNS
        ]
        .copy()
    )


    sampling_rows.append(
        {
            "parent_asin":
                product,

            "all_available_reviews":
                len(
                    product_all
                ),

            "valid_timestamp_reviews":
                len(
                    valid
                ),

            "selected_raw_reviews":
                len(
                    sample
                ),

            "earliest_available_review":
                str(
                    valid[
                        "_sample_date"
                    ]
                    .min()
                ),

            "latest_available_review":
                str(
                    valid[
                        "_sample_date"
                    ]
                    .max()
                ),

            "earliest_selected_review":
                str(
                    sample[
                        "_sample_date"
                    ]
                    .min()
                ),

            "latest_selected_review":
                str(
                    sample[
                        "_sample_date"
                    ]
                    .max()
                ),
        }
    )


laptop_pilot_raw = (
    pd.concat(
        sampled_parts,
        ignore_index=True,
    )
)


temporal_sampling_df = (
    pd.DataFrame(
        sampling_rows
    )
)


assert (
    len(
        laptop_pilot_raw
    )
    ==
    (
        TARGET_PILOT_PRODUCTS
        *
        FINAL_RAW_REVIEWS_PER_PRODUCT
    )
)


laptop_pilot_raw.to_parquet(
    RAW_PILOT_PATH,
    index=False,
)


temporal_sampling_df.to_csv(
    TEMPORAL_SAMPLING_PATH,
    index=False,
)


print(
    "Final raw pilot rows:",
    len(
        laptop_pilot_raw
    ),
)


display(
    temporal_sampling_df
)


Final raw pilot rows: 1000


,parent_asin,all_available_reviews,valid_timestamp_reviews,selected_raw_reviews,earliest_available_review,latest_available_review,earliest_selected_review,latest_selected_review
0,B009T3EYHO,5117,5117,100,2012-10-24 22:06:32+00:00,2023-03-10 06:01:49.727000+00:00,2012-10-24 22:06:32+00:00,2023-03-10 06:01:49.727000+00:00
1,B01K1IO3QW,4056,4056,100,2016-09-04 13:25:14+00:00,2023-07-11 15:40:57.062000+00:00,2016-09-04 13:25:14+00:00,2023-07-11 15:40:57.062000+00:00
2,B07GM2J11Q,1854,1854,100,2018-10-23 23:12:16.104000+00:00,2023-08-31 18:30:13.869000+00:00,2018-10-23 23:12:16.104000+00:00,2023-08-31 18:30:13.869000+00:00
3,B07N6S4SY1,642,642,100,2019-06-20 06:34:47.622000+00:00,2023-05-10 00:13:42.131000+00:00,2019-06-20 06:34:47.622000+00:00,2023-05-10 00:13:42.131000+00:00
4,B081V6W99V,827,827,100,2020-10-11 06:28:39.070000+00:00,2023-08-17 15:27:24.789000+00:00,2020-10-11 06:28:39.070000+00:00,2023-08-17 15:27:24.789000+00:00
5,B0869L1326,557,557,100,2020-05-30 22:25:13.337000+00:00,2023-07-31 16:54:30.667000+00:00,2020-05-30 22:25:13.337000+00:00,2023-07-31 16:54:30.667000+00:00
6,B08GG9QKX6,609,609,100,2020-03-28 04:32:52.026000+00:00,2023-02-25 21:04:12.630000+00:00,2020-03-28 04:32:52.026000+00:00,2023-02-25 21:04:12.630000+00:00
7,B096JQY4YR,530,530,100,2020-04-29 20:15:07.472000+00:00,2023-08-01 17:29:50.834000+00:00,2020-04-29 20:15:07.472000+00:00,2023-08-01 17:29:50.834000+00:00
8,B0C2ZMJW53,5053,5053,100,2019-06-16 21:52:21.503000+00:00,2023-08-22 15:47:07.684000+00:00,2019-06-16 21:52:21.503000+00:00,2023-08-22 15:47:07.684000+00:00
9,B0C7H1XD7V,1187,1187,100,2020-05-24 20:15:39.171000+00:00,2023-08-26 17:01:12.131000+00:00,2020-05-24 20:15:39.171000+00:00,2023-08-26 17:01:12.131000+00:00


## 11. Smartphone-Consistent Cleaning and Stable IDs


In [12]:
raw_df = pd.read_parquet(
    RAW_PILOT_PATH
)


audit = {
    "raw_rows":
        len(
            raw_df
        ),

    "missing_text":
        int(
            raw_df[
                "text"
            ]
            .isna()
            .sum()
        ),

    "empty_text":
        int(
            raw_df[
                "text"
            ]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        ),

    "duplicate_product_text":
        int(
            raw_df
            .duplicated(
                subset=[
                    "parent_asin",
                    "text",
                ]
            )
            .sum()
        ),

    "missing_rating":
        int(
            raw_df[
                "rating"
            ]
            .isna()
            .sum()
        ),

    "missing_helpful_vote":
        int(
            raw_df[
                "helpful_vote"
            ]
            .isna()
            .sum()
        ),

    "missing_timestamp":
        int(
            raw_df[
                "timestamp"
            ]
            .isna()
            .sum()
        ),

    "missing_verified_purchase":
        int(
            raw_df[
                "verified_purchase"
            ]
            .isna()
            .sum()
        ),
}


audit_df = pd.DataFrame(
    {
        "metric":
            list(
                audit.keys()
            ),

        "value":
            list(
                audit.values()
            ),
    }
)


display(
    audit_df
)


DATA_QUALITY_PATH = (
    RESULTS_DIR
    /
    "laptop_data_quality_audit.csv"
)


audit_df.to_csv(
    DATA_QUALITY_PATH,
    index=False,
)


df = raw_df.copy()


df[
    "text"
] = (
    df[
        "text"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)


df[
    "title"
] = (
    df[
        "title"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)


df[
    "rating"
] = pd.to_numeric(
    df[
        "rating"
    ],
    errors="coerce",
)


df[
    "helpful_vote"
] = pd.to_numeric(
    df[
        "helpful_vote"
    ],
    errors="coerce",
)


df[
    "timestamp"
] = pd.to_numeric(
    df[
        "timestamp"
    ],
    errors="coerce",
)


before = len(
    df
)


df = df[
    df[
        "text"
    ].ne("")
].copy()


df = df[
    df[
        "rating"
    ].between(
        1,
        5,
        inclusive="both",
    )
].copy()


df = df[
    df[
        "helpful_vote"
    ]
    .fillna(-1)
    >=
    0
].copy()


df = df[
    df[
        "timestamp"
    ]
    .notna()
].copy()


df = df.drop_duplicates(
    subset=[
        "parent_asin",
        "text",
    ],
    keep="first",
).copy()


df[
    "review_date"
] = pd.to_datetime(
    df[
        "timestamp"
    ],
    unit="ms",
    errors="coerce",
    utc=True,
)


df = df[
    df[
        "review_date"
    ]
    .notna()
].copy()


def make_review_id(
    row
):

    key = "||".join(
        [
            str(
                row.get(
                    "parent_asin",
                    "",
                )
            ),

            str(
                row.get(
                    "asin",
                    "",
                )
            ),

            str(
                row.get(
                    "timestamp",
                    "",
                )
            ),

            str(
                row.get(
                    "title",
                    "",
                )
            ),

            str(
                row.get(
                    "text",
                    "",
                )
            ),
        ]
    )


    return hashlib.sha256(
        key.encode(
            "utf-8"
        )
    ).hexdigest()[:16]


df[
    "review_id"
] = df.apply(
    make_review_id,
    axis=1,
)


if (
    df[
        "review_id"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Duplicate stable review IDs."
    )


df[
    "review_length_words"
] = (
    df[
        "text"
    ]
    .str.split()
    .str.len()
    .astype(int)
)


df[
    "review_length_chars"
] = (
    df[
        "text"
    ]
    .str.len()
    .astype(int)
)


df[
    "verified_purchase"
] = (
    df[
        "verified_purchase"
    ]
    .fillna(False)
    .astype(bool)
)


print(
    "Rows before cleaning:",
    before,
)


print(
    "Rows after cleaning:",
    len(
        df
    ),
)


print(
    "Rows removed:",
    before
    -
    len(
        df
    ),
)


,metric,value
0,raw_rows,1000
1,missing_text,0
2,empty_text,2
3,duplicate_product_text,1
4,missing_rating,0
5,missing_helpful_vote,0
6,missing_timestamp,0
7,missing_verified_purchase,0


Rows before cleaning: 1000
Rows after cleaning: 997
Rows removed: 3


## 12. EDA and Candidate Usefulness Targets


In [13]:
zero_helpful_count = int(
    (
        df[
            "helpful_vote"
        ]
        ==
        0
    )
    .sum()
)


zero_helpful_pct = float(
    (
        df[
            "helpful_vote"
        ]
        ==
        0
    )
    .mean()
    *
    100
)


raw_skew = float(
    skew(
        df[
            "helpful_vote"
        ],
        bias=False,
    )
)


df[
    "log_helpful"
] = np.log1p(
    df[
        "helpful_vote"
    ]
)


log_skew = float(
    skew(
        df[
            "log_helpful"
        ],
        bias=False,
    )
)


verified_pct = float(
    df[
        "verified_purchase"
    ]
    .mean()
    *
    100
)


earliest_review = (
    df[
        "review_date"
    ]
    .min()
)


latest_review = (
    df[
        "review_date"
    ]
    .max()
)


reference_date = (
    latest_review
    +
    pd.Timedelta(
        days=1
    )
)


df[
    "age_days"
] = (
    (
        reference_date
        -
        df[
            "review_date"
        ]
    )
    .dt
    .total_seconds()
    .div(
        86400
    )
    .clip(
        lower=1
    )
)


age_helpful_corr = spearmanr(
    df[
        "age_days"
    ],
    df[
        "helpful_vote"
    ],
    nan_policy="omit",
)


length_helpful_corr = spearmanr(
    df[
        "review_length_words"
    ],
    df[
        "helpful_vote"
    ],
    nan_policy="omit",
)


# Exact smartphone target formulations.

df[
    "raw_helpful"
] = (
    df[
        "helpful_vote"
    ]
    .astype(float)
)


df[
    "log_helpful"
] = np.log1p(
    df[
        "helpful_vote"
    ]
)


df[
    "product_relative"
] = (

    df.groupby(
        "parent_asin"
    )[
        "log_helpful"
    ]

    .rank(
        method="average",
        pct=True,
    )
)


effective_age_days = (

    df[
        "age_days"
    ]

    .clip(
        lower=30
    )
)


df[
    "age_adjusted"
] = (
    df[
        "helpful_vote"
    ]
    /
    effective_age_days
    *
    365.25
)


target_columns = [
    "raw_helpful",
    "log_helpful",
    "product_relative",
    "age_adjusted",
]


def safe_skew(
    series
):

    values = (
        series
        .dropna()
    )


    if (
        len(
            values
        )
        <
        3
        or
        values.nunique()
        <=
        1
    ):

        return np.nan


    return float(
        skew(
            values,
            bias=False,
        )
    )


target_rows = []


for target in target_columns:

    values = (
        df[
            target
        ]
        .dropna()
    )


    target_rows.append(
        {
            "target":
                target,

            "count":
                len(
                    values
                ),

            "unique_values":
                values.nunique(),

            "zero_pct":
                (
                    values
                    ==
                    0
                )
                .mean()
                *
                100,

            "mean":
                values.mean(),

            "median":
                values.median(),

            "std":
                values.std(),

            "min":
                values.min(),

            "max":
                values.max(),

            "skewness":
                safe_skew(
                    values
                ),
        }
    )


target_diagnostics_df = (
    pd.DataFrame(
        target_rows
    )
)


TARGET_DIAGNOSTICS_PATH = (
    RESULTS_DIR
    /
    "laptop_target_diagnostics.csv"
)


target_diagnostics_df.to_csv(
    TARGET_DIAGNOSTICS_PATH,
    index=False,
)


display(
    target_diagnostics_df
)


raw_target_skew = float(

    target_diagnostics_df

    .set_index(
        "target"
    )

    .loc[
        "raw_helpful",
        "skewness",
    ]
)


log_target_skew = float(

    target_diagnostics_df

    .set_index(
        "target"
    )

    .loc[
        "log_helpful",
        "skewness",
    ]
)


log_reduces_skew = (
    abs(
        log_target_skew
    )
    <
    abs(
        raw_target_skew
    )
)


PRELIMINARY_USEFULNESS_TARGET = (
    "log_helpful"
)


target_status = (

    "supported_for_pp1_cross_category_baseline"

    if log_reduces_skew

    else

    "requires_review_before_laptop_function_1"
)


print(
    "Cleaned reviews:",
    len(
        df
    ),
)


print(
    "Products:",
    df[
        "parent_asin"
    ].nunique(),
)


print(
    f"Zero helpful: "
    f"{zero_helpful_pct:.2f}%"
)


print(
    f"Raw helpful skewness: "
    f"{raw_skew:.3f}"
)


print(
    f"Log helpful skewness: "
    f"{log_skew:.3f}"
)


print(
    "Preliminary target:",
    PRELIMINARY_USEFULNESS_TARGET,
)


,target,count,unique_values,zero_pct,mean,median,std,min,max,skewness
0,raw_helpful,997,31,61.885657,13.111334,0.000,270.906693,0.00,8384.000000,29.847648
1,log_helpful,997,31,61.885657,0.470013,0.000,0.856366,0.00,9.034200,3.794552
2,product_relative,997,51,0.000000,0.505015,0.345,0.249772,0.25,1.000000,0.685704
3,age_adjusted,997,381,61.885657,2.022271,0.000,27.329899,0.00,772.545924,24.201673


Cleaned reviews: 997
Products: 10
Zero helpful: 61.89%
Raw helpful skewness: 29.848
Log helpful skewness: 3.795
Preliminary target: log_helpful


## 13. Product-Level EDA


In [14]:
product_summary = (

    df.groupby(
        "parent_asin"
    )

    .agg(

        review_count=(
            "text",
            "size",
        ),

        avg_rating=(
            "rating",
            "mean",
        ),

        mean_helpful_vote=(
            "helpful_vote",
            "mean",
        ),

        median_helpful_vote=(
            "helpful_vote",
            "median",
        ),

        max_helpful_vote=(
            "helpful_vote",
            "max",
        ),

        zero_helpful_pct=(
            "helpful_vote",
            lambda x:
                (
                    x
                    ==
                    0
                )
                .mean()
                *
                100,
        ),

        verified_purchase_pct=(
            "verified_purchase",
            lambda x:
                x.mean()
                *
                100,
        ),

        avg_review_length_words=(
            "review_length_words",
            "mean",
        ),

        earliest_review=(
            "review_date",
            "min",
        ),

        latest_review=(
            "review_date",
            "max",
        ),
    )

    .reset_index()

    .sort_values(
        "review_count",
        ascending=False,
    )
)


display(
    product_summary
)


PRODUCT_SUMMARY_PATH = (
    RESULTS_DIR
    /
    "laptop_product_summary.csv"
)


product_summary.to_csv(
    PRODUCT_SUMMARY_PATH,
    index=False,
)


,parent_asin,review_count,avg_rating,mean_helpful_vote,median_helpful_vote,max_helpful_vote,zero_helpful_pct,verified_purchase_pct,avg_review_length_words,earliest_review,latest_review
0,B009T3EYHO,100,4.160000,86.710000,1.0,8384,49.000000,82.000000,129.180000,2012-10-24 22:06:32+00:00,2023-03-10 06:01:49.727000+00:00
3,B07N6S4SY1,100,3.460000,11.090000,0.0,788,61.000000,99.000000,52.170000,2019-06-20 06:34:47.622000+00:00,2023-05-10 00:13:42.131000+00:00
9,B0C7H1XD7V,100,3.390000,1.880000,0.0,77,57.000000,97.000000,89.310000,2020-05-24 20:15:39.171000+00:00,2023-08-26 17:01:12.131000+00:00
4,B081V6W99V,100,3.610000,0.950000,0.0,16,61.000000,96.000000,52.080000,2020-10-11 06:28:39.070000+00:00,2023-08-17 15:27:24.789000+00:00
5,B0869L1326,100,3.780000,0.700000,0.0,9,68.000000,97.000000,49.090000,2020-05-30 22:25:13.337000+00:00,2023-07-31 16:54:30.667000+00:00
6,B08GG9QKX6,100,3.930000,8.160000,0.0,434,68.000000,95.000000,44.400000,2020-03-28 04:32:52.026000+00:00,2023-02-25 21:04:12.630000+00:00
8,B0C2ZMJW53,100,3.920000,0.860000,0.0,12,61.000000,98.000000,65.000000,2019-06-16 21:52:21.503000+00:00,2023-08-22 15:47:07.684000+00:00
1,B01K1IO3QW,99,3.555556,1.131313,0.0,19,59.595960,95.959596,72.545455,2016-09-04 13:25:14+00:00,2023-07-11 15:40:57.062000+00:00
2,B07GM2J11Q,99,4.090909,14.949495,0.0,1414,62.626263,94.949495,48.282828,2018-10-23 23:12:16.104000+00:00,2023-08-31 18:30:13.869000+00:00
7,B096JQY4YR,99,3.777778,4.494949,0.0,206,71.717172,95.959596,37.020202,2020-04-29 20:15:07.472000+00:00,2023-08-01 17:29:50.834000+00:00


## 14. Per-Product Chronological 70/15/15 Split


In [15]:
def chronological_split_group(
    group,
):

    group = (
        group
        .sort_values(
            "review_date"
        )
        .copy()
    )


    n = len(
        group
    )


    train_end = max(
        1,
        int(
            math.floor(
                n
                *
                0.70
            )
        ),
    )


    val_end = max(
        train_end
        +
        1,

        int(
            math.floor(
                n
                *
                0.85
            )
        ),
    )


    val_end = min(
        val_end,
        n,
    )


    split = np.array(
        [
            "test"
        ]
        *
        n,
        dtype=object,
    )


    split[
        :train_end
    ] = "train"


    split[
        train_end:
        val_end
    ] = "validation"


    if n == 1:

        split[:] = "train"


    elif n == 2:

        split[0] = "train"

        split[1] = "test"


    group[
        "split"
    ] = split


    return group


split_parts = []


for (
    _,
    group,
) in df.groupby(
    "parent_asin",
    sort=False,
):

    split_parts.append(
        chronological_split_group(
            group
        )
    )


split_df = pd.concat(
    split_parts,
    ignore_index=True,
)


display(
    split_df[
        "split"
    ]
    .value_counts()
    .to_frame(
        "review_count"
    )
)


display(
    pd.crosstab(
        split_df[
            "parent_asin"
        ],
        split_df[
            "split"
        ],
    )
)


chronology_rows = []


for (
    product,
    group,
) in split_df.groupby(
    "parent_asin"
):

    train_dates = group.loc[
        group[
            "split"
        ]
        ==
        "train",
        "review_date",
    ]


    val_dates = group.loc[
        group[
            "split"
        ]
        ==
        "validation",
        "review_date",
    ]


    test_dates = group.loc[
        group[
            "split"
        ]
        ==
        "test",
        "review_date",
    ]


    valid = True


    if (
        not train_dates.empty
        and
        not val_dates.empty
    ):

        valid = (
            valid
            and
            train_dates.max()
            <=
            val_dates.min()
        )


    if (
        not val_dates.empty
        and
        not test_dates.empty
    ):

        valid = (
            valid
            and
            val_dates.max()
            <=
            test_dates.min()
        )


    chronology_rows.append(
        {
            "parent_asin":
                product,

            "chronology_valid":
                bool(
                    valid
                ),
        }
    )


chronology_check_df = (
    pd.DataFrame(
        chronology_rows
    )
)


display(
    chronology_check_df
)


,review_count
split,
train,697
validation,150
test,150


split,test,train,validation
parent_asin,,,
B009T3EYHO,15,70,15
B01K1IO3QW,15,69,15
B07GM2J11Q,15,69,15
B07N6S4SY1,15,70,15
B081V6W99V,15,70,15
B0869L1326,15,70,15
B08GG9QKX6,15,70,15
B096JQY4YR,15,69,15
B0C2ZMJW53,15,70,15


,parent_asin,chronology_valid
0,B009T3EYHO,True
1,B01K1IO3QW,True
2,B07GM2J11Q,True
3,B07N6S4SY1,True
4,B081V6W99V,True
5,B0869L1326,True
6,B08GG9QKX6,True
7,B096JQY4YR,True
8,B0C2ZMJW53,True
9,B0C7H1XD7V,True


## 15. Save Final Laptop Pilot and Evidence


In [16]:
columns_to_save = [
    "review_id",
    "rating",
    "title",
    "text",
    "asin",
    "parent_asin",
    "timestamp",
    "helpful_vote",
    "verified_purchase",
    "review_date",
    "review_length_words",
    "review_length_chars",
    "age_days",
    "raw_helpful",
    "log_helpful",
    "product_relative",
    "age_adjusted",
]


cleaned_for_save = (
    df[
        columns_to_save
    ]
    .copy()
)


cleaned_for_save.to_parquet(
    CLEANED_PILOT_PATH,
    index=False,
)


SPLIT_PATH = (
    RESULTS_DIR
    /
    "laptop_chronological_split.csv"
)


split_df[
    [
        "review_id",
        "asin",
        "parent_asin",
        "review_date",
        "split",
    ]
].to_csv(
    SPLIT_PATH,
    index=False,
)


EDA_SUMMARY_PATH = (
    RESULTS_DIR
    /
    "laptop_eda_summary.json"
)


eda_summary = {
    "pipeline_version":
        PIPELINE_VERSION,

    "raw_rows":
        int(
            len(
                raw_df
            )
        ),

    "cleaned_rows":
        int(
            len(
                df
            )
        ),

    "rows_removed":
        int(
            len(
                raw_df
            )
            -
            len(
                df
            )
        ),

    "products":
        int(
            df[
                "parent_asin"
            ]
            .nunique()
        ),

    "zero_helpful_count":
        int(
            zero_helpful_count
        ),

    "zero_helpful_pct":
        float(
            zero_helpful_pct
        ),

    "raw_helpful_skewness":
        float(
            raw_skew
        ),

    "log_helpful_skewness":
        float(
            log_skew
        ),

    "earliest_review":
        str(
            earliest_review
        ),

    "latest_review":
        str(
            latest_review
        ),

    "verified_purchase_pct":
        float(
            verified_pct
        ),

    "age_helpful_spearman_rho":
        float(
            age_helpful_corr.statistic
        ),

    "age_helpful_spearman_p":
        float(
            age_helpful_corr.pvalue
        ),

    "length_helpful_spearman_rho":
        float(
            length_helpful_corr.statistic
        ),

    "length_helpful_spearman_p":
        float(
            length_helpful_corr.pvalue
        ),

    "preliminary_usefulness_target":
        PRELIMINARY_USEFULNESS_TARGET,

    "target_evidence_status":
        target_status,

    "metadata_laptop_candidates":
        int(
            laptop_candidates[
                "parent_asin"
            ]
            .nunique()
        ),

    "eligible_products":
        int(
            len(
                eligible_products
            )
        ),

    "final_product_signature":
        final_product_signature,
}


EDA_SUMMARY_PATH.write_text(
    json.dumps(
        eda_summary,
        indent=2,
    ),
    encoding="utf-8",
)


CONFIG_PATH = (
    RESULTS_DIR
    /
    "laptop_pilot_config.json"
)


config = {
    "pipeline_version":
        PIPELINE_VERSION,

    "dataset_id":
        DATASET_ID,

    "metadata_revision":
        META_REVISION,

    "review_revision":
        REVIEW_REVISION,

    "selected_parent_asins":
        sorted(
            FINAL_LAPTOP_ASINS
        ),

    "product_selection_uses_helpful_votes":
        False,

    "final_sampling_policy":
        (
            "Full final-product review extraction "
            "followed by deterministic chronological "
            "time-spread selection of 100 raw reviews/product."
        ),

    "duplicate_definition":
        [
            "parent_asin",
            "text",
        ],

    "stable_review_id":
        (
            "SHA256(parent_asin||asin||timestamp||"
            "title||text)[:16]"
        ),

    "timestamp_unit":
        "ms",

    "preliminary_target":
        PRELIMINARY_USEFULNESS_TARGET,

    "split_policy":
        (
            "Per-product chronological "
            "70/15/15"
        ),
}


CONFIG_PATH.write_text(
    json.dumps(
        config,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "Saved raw pilot:",
    RAW_PILOT_PATH,
)


print(
    "Saved cleaned pilot:",
    CLEANED_PILOT_PATH,
)


print(
    "Saved EDA summary:",
    EDA_SUMMARY_PATH,
)


Saved raw pilot: D:\Research\Repository\J26-DS-349\component_2\data\pilot\laptop_pilot_raw.parquet
Saved cleaned pilot: D:\Research\Repository\J26-DS-349\component_2\data\pilot\laptop_pilot_cleaned.parquet
Saved EDA summary: D:\Research\Repository\J26-DS-349\component_2\results\eda\laptop\laptop_eda_summary.json


## 16. Final Validation


In [17]:
assert (
    len(
        completed_discovery_shards
    )
    ==
    len(
        review_repo_files
    )
)


assert (
    len(
        completed_final_shards
    )
    ==
    len(
        review_repo_files
    )
)


assert (
    len(
        FINAL_LAPTOP_ASINS
    )
    ==
    10
)


assert (
    len(
        raw_df
    )
    ==
    1000
)


assert (
    raw_df[
        "parent_asin"
    ]
    .astype(str)
    .value_counts()
    .eq(
        100
    )
    .all()
)


assert (
    df[
        "parent_asin"
    ]
    .nunique()
    ==
    10
)


assert (
    df[
        "review_id"
    ]
    .nunique()
    ==
    len(
        df
    )
)


assert not df.duplicated(
    subset=[
        "parent_asin",
        "text",
    ]
).any()


assert not df[
    "text"
].astype(str).str.strip().eq(
    ""
).any()


assert df[
    "rating"
].between(
    1,
    5,
    inclusive="both",
).all()


assert (
    df[
        "helpful_vote"
    ]
    >=
    0
).all()


assert (
    chronology_check_df[
        "chronology_valid"
    ]
    .all()
)


required_splits = {
    "train",
    "validation",
    "test",
}


for (
    product,
    group,
) in split_df.groupby(
    "parent_asin"
):

    assert (
        required_splits
        .issubset(
            set(
                group[
                    "split"
                ]
            )
        )
    )


print()

print(
    "Laptop pilot dataset and EDA validation passed."
)


print(
    "Raw reviews:",
    len(
        raw_df
    ),
)


print(
    "Cleaned reviews:",
    len(
        df
    ),
)


print(
    "Laptop products:",
    df[
        "parent_asin"
    ].nunique(),
)


print(
    "Minimum cleaned reviews/product:",
    int(
        df.groupby(
            "parent_asin"
        )
        .size()
        .min()
    ),
)


print(
    "Zero helpful percentage:",
    round(
        zero_helpful_pct,
        2,
    ),
)


print(
    "Raw helpful skewness:",
    round(
        raw_skew,
        3,
    ),
)


print(
    "Log helpful skewness:",
    round(
        log_skew,
        3,
    ),
)


print(
    "Preliminary usefulness target:",
    PRELIMINARY_USEFULNESS_TARGET,
)


print(
    "Chronology valid:",
    bool(
        chronology_check_df[
            "chronology_valid"
        ]
        .all()
    ),
)


print()

print(
    "MANUAL CHECK STILL REQUIRED:"
)


print(
    "Confirm the frozen 10 titles are "
    "actual laptops/notebooks before "
    "closing Milestone 10."
)



Laptop pilot dataset and EDA validation passed.
Raw reviews: 1000
Cleaned reviews: 997
Laptop products: 10
Minimum cleaned reviews/product: 99
Zero helpful percentage: 61.89
Raw helpful skewness: 29.848
Log helpful skewness: 3.795
Preliminary usefulness target: log_helpful
Chronology valid: True

MANUAL CHECK STILL REQUIRED:
Confirm the frozen 10 titles are actual laptops/notebooks before closing Milestone 10.
